# Phase 9 — Error Analysis Notebook

**Project:** AI-Powered Customer Feedback Intelligence System  
**Goal:** Inspect actual misclassified customer reviews to identify recurring error patterns, rating/text proxy mismatches, and linguistic edge cases.

### Error Analysis Objectives:
1. Compute error rates for Naive Bayes, Logistic Regression, and Linear SVM.
2. Analyze class-wise error distributions (Why `Neutral` has ~76% error rate).
3. Examine specific misclassified examples (Mixed sentiment, sarcasm, negation).
4. Compare review length impact on model accuracy.
5. Formulate actionable future improvements (BERT, Aspect-Based Sentiment Analysis).

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split

# Load preprocessed clean text
df = pd.read_csv('../data/processed/reviews_cleaned.csv')
df['clean_text'] = df['clean_text'].fillna('')
df['word_count'] = df['clean_text'].apply(lambda s: len(s.split()))

_, X_test_raw, _, y_test = train_test_split(
    df['clean_text'], df['Sentiment'], test_size=0.20, random_state=42, stratify=df['Sentiment']
)
df_test = df.iloc[X_test_raw.index].copy()

tfidf = joblib.load('../models/tfidf_vectorizer.pkl')
lr_model = joblib.load('../models/logistic_regression_model.pkl')

df_test['pred_lr'] = lr_model.predict(tfidf.transform(X_test_raw))
df_errors = df_test[df_test['Sentiment'] != df_test['pred_lr']]

print(f'Total Test Reviews : {len(df_test):,}')
print(f'Total Error Reviews: {len(df_errors):,} ({len(df_errors)/len(df_test)*100:.2f}%)')
display(df_errors[['Score', 'Sentiment', 'pred_lr', 'Text']].head(5))